In [1]:
import pandas as pd
import wget
import os
from datetime import datetime as dt

In [2]:
fn = "FuelPriceHistory"
dir_in = "../source_data/"
dir_out = "../parsed_data/"

In [3]:
url = "http://ec.europa.eu/energy/observatory/reports/Oil_Bulletin_Prices_History.xls"

In [4]:
wget.download(url,dir_in+fn) 

100% [........................................................................] 13222912 / 13222912

'../source_data/FuelPriceHistory'

In [5]:
df = pd.read_excel(dir_in+fn,sheet_name="Prices with taxes, per CTR", 
                   thousands=",",usecols="A:E",index_col=[0,1]).dropna().reset_index()

In [6]:
df = df.rename(columns={"level_0":"country","level_1":"date","Unnamed: 2":"Exchange_rate", "Unnamed: 3" : "Super", "Unnamed: 4":"Diesel"})

In [7]:
df = df[df.Exchange_rate != "ExchangeRateTo €"] 
df.date = pd.to_datetime(df.date)
df.Super = df.Super.astype("float")
df.Diesel = df.Diesel.astype("float")
#df.Exchange_rate = df.Exchange_rate.astype("float")
#df.Super = df.Super * df.Exchange_rate
#df.Diesel = df.Diesel * df.Exchange_rate
df = df.drop(columns={"Exchange_rate"})
df['year'] = df.date.dt.year
df['Mean'] = (df.Diesel + df.Super) / 2
df.head(1)

,country,date,Super,Diesel,year,Mean
1,AT,2018-09-03,1310.0,1250.0,2018,1280.0


In [8]:
df_year = df.groupby(['country','year']).mean().reset_index()
df_year.head(1)

,country,year,Super,Diesel,Mean
0,AT,2005,1033.530612,947.693878,990.612245


In [9]:
df_2017 = df_year[df_year.year == 2017]
df_2017 = df_2017.drop(columns='year')
df_2017.head(1)

,country,Super,Diesel,Mean
12,AT,1177.44,1105.32,1141.38


In [10]:
df_2017.to_csv(dir_out+'passengervehicles_fuel_prices.csv', encoding="utf-8",index=False)